# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset schema is available at the following Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and explore basic information about the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Obtain metadata as dict
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}\n\n{metadata_json['description']}")
print(f"\nPublished: {metadata_json['datePublished']}, Version: {metadata_json.get('version', '')}")

## 2. Data Overview
Review available record sets and fields.

In Croissant, record sets, fields, and columns are referenced by their unique `@id` values. Let's enumerate them for this dataset.

In [ ]:
# List record sets and their fields by @id
print("Available record sets (with @id):")
for rs in dataset.record_sets():
    print(f"- @id: {rs.id} | name: {rs.name}")

# For illustration, get fields/columns for first record set
record_sets = list(dataset.record_sets())
if record_sets:
    main_record_set = record_sets[0]
    print(f"\nFields and columns in record set @id={main_record_set.id}:")
    for field in main_record_set.fields:
        print(f"  Field @id: {field.id} | name: {field.name} | dataType: {field.data_type if hasattr(field, 'data_type') else ''}")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"    Column @id: {col.id} | name: {col.name}")

## 3. Data Extraction
We will extract all available record sets. You can refer to entities by their `@id` when loading data.

Here, we demonstrate how to extract all records from each record set and assemble them as pandas DataFrames for downstream analysis.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets()]
print("Record set @id list:")
pprint.pprint(record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# Display columns of main record set
if record_set_ids:
    main_id = record_set_ids[0]
    print(f"\nColumns in primary record set ({main_id}):\n", dataframes[main_id].columns.tolist())
    dataframes[main_id].head()

## 4. Exploratory Data Analysis (EDA)
We can now process and analyze the data. As an example, let's work with a numeric field like the total GAD-7 score, which is a common mental health screening metric.

In [ ]:
# Select record set and corresponding numeric field (by @id)
# Common field ids could be e.g. 'cr:GAD7_total_score'.
# Let's enumerate columns for all record sets to find candidate numeric fields.
numeric_field = None
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

print(f"Columns available in record set {main_record_set_id}:")
print(df.columns.tolist())

# Try to select a GAD-7 total score field, or fallback to a numeric field.
for col in df.columns:
    if 'gad7' in col.lower() and 'score' in col.lower():
        numeric_field = col
        break
if numeric_field is None:
    for col in df.columns:
        # Look for a typical numeric field
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

print(f"\nSelected numeric field for analysis: {numeric_field}")

# Example: filter for high GAD-7 scores (possible anxiety): threshold = 10
threshold = 10
if numeric_field is not None:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (count={len(filtered_df)}):")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print("\nFirst few normalized values:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical demographic field, e.g. 'gender' or 'level_of_education'
    group_candidates = [c for c in df.columns if 'gender' in c.lower() or 'education' in c.lower()]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nMean {numeric_field} by {group_field} for filtered records:")
        display(grouped)
else:
    print("No numeric field detected for EDA.")

## 5. Visualization
Visualize the distribution of GAD-7 (or selected numeric field) scores, and breakdown by demographic attribute if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and not df[numeric_field].isnull().all():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15, color='teal')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # Optional: boxplot by demographic group
    if group_candidates:
        group_field = group_candidates[0]
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
We explored the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library. We:
- Loaded and inspected dataset metadata
- Listed record sets, fields, and columns by their `@id`
- Loaded records into pandas DataFrames
- Performed basic filtering and normalization on a sample numeric field (e.g., GAD-7 score)
- Grouped and visualized the field by a categorical demographic variable

For deeper insight, you can extend the analysis with more fields, advanced visualizations, and domain-specific statistical analyses.